# Merge Entries with Claims

After the work I've done to extract data from TEI XML and convert it to JSON, I need to merge the apparatus entries with their apparatus claims. I confess that I needed to consult ChatGPT for help with this.

In [1]:
import json
from pathlib import Path
from collections import defaultdict

First I should remove all the apparatus_claims that hold no information—a leftover from a previous operation.

In [9]:
with open("../converted-json/calpSicPoem1-target-array.json", "r") as f:
    data = json.load(f)
    claims = [claim for claim in data]
    original_count = len(claims)

    claims = [
        claim
        for claim in claims
        if not (
            claim.get("reading_type") == "lemma" and claim.get("supporting_evidence") == []
        )
    ]

    print(f"Removed {original_count - len(claims)} erroneous claims.")

Removed 169 erroneous claims.


In [30]:
with open("../converted-json/apparatus_claims.json", "w") as f:
    json.dump(claims, f, indent=4, ensure_ascii=False)

In [31]:
# ChatGPT wrote this function to merge apparatus entries with their associated claims.
def merge_apparatus(entries_file, claims_file, output_file):
    """
    Merge apparatus entries with their associated claims.
    
    Args:
    - entries_file: Path to the JSON file containing apparatus entries.
    - claims_file: Path to the JSON file containing claims.
    - output_file: Path to the output JSON file where the merged data will be saved.
    
    Returns:
    - A list containing the merged apparatus entries and claims.
    """

    # Load both JSON files
    with Path(entries_file).open("r", encoding="utf-8") as f:
        entries = json.load(f)

    with Path(claims_file).open("r", encoding="utf-8") as f:
        claims = json.load(f)

    # Group claims by apparatus_entry_id
    claims_by_entry = defaultdict(list)

    for claim in claims:
        claims_by_entry[claim["apparatus_entry_id"]].append(claim)

    # Assemble the combined list
    merged = []

    for entry in entries:
        merged.append(entry)

        # Retrieve claims associated with this entry
        associated_claims = claims_by_entry.get(entry["id"], [])

        # Sort claims by reporting_rank
        associated_claims.sort(
            key=lambda claim: claim.get("reporting_rank") or float("inf")
        )

        merged.extend(associated_claims)

    entry_ids = {entry["id"] for entry in entries}

    orphaned_claims = [
        claim for claim in claims if claim["apparatus_entry_id"] not in entry_ids
    ]

    if orphaned_claims:
        for claim in orphaned_claims:
            print(
                f"Orphaned claim found: {claim['id']} with cts_id {claim['target']} and apparatus_entry_id {claim['apparatus_entry_id']}"
            )
        raise ValueError(f"{len(orphaned_claims)} claims have no matching apparatus entry.")

    # Write the merged JSON
    with Path(output_file).open("w", encoding="utf-8") as f:
        json.dump(merged, f, indent=4, ensure_ascii=False)

    return merged

In [2]:
entries = "../converted-json/apparatus_entries.json"
claims_file = "../converted-json/apparatus_claims.json"
output_file = "../converted-json/merged-apparatus-entries-and-claims.json"

In [36]:
merged = merge_apparatus(entries, claims_file, output_file)

In [3]:
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=4, ensure_ascii=False)

NameError: name 'merged' is not defined